In [1]:
import os
import numpy as np
import SimpleITK as sitk
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
COLNAME_EN_MAP = {
    "subj_names": "Subject ID",
    "subj_key": "Subject Key",
    "gestational_age": "Gestational Age (weeks)",

    "wm_area_l_list": "Left WM Surface Area (mm²)",
    "wm_area_r_list": "Right WM Surface Area (mm²)",
    "cortex_area_l_list": "Left Cortex Surface Area (mm²)",
    "cortex_area_r_list": "Right Cortex Surface Area (mm²)",

    # 下面这些 volume 的单位文字会在函数里按 y_scale 自动替换
    "wm_vol_l_list": "Left WM Volume ({unit})",
    "wm_vol_r_list": "Right WM Volume ({unit})",
    "cortex_vol_l_list": "Left Cortex Volume ({unit})",
    "cortex_vol_r_list": "Right Cortex Volume ({unit})",
    "left_hemispheres_vol": "Left Hemisphere Volume ({unit})",
    "right_hemispheres_vol": "Right Hemisphere Volume ({unit})",

    "脑脊液": "Cerebrospinal Fluid (CSF) Volume ({unit})",
    "皮质（左）": "Left Cortex Volume ({unit})",
    "皮质（右）": "Right Cortex Volume ({unit})",
    "白质（左）": "Left White Matter Volume ({unit})",
    "白质（右）": "Right White Matter Volume ({unit})",
    "侧脑室（左）": "Left Lateral Ventricle Volume ({unit})",
    "侧脑室（右）": "Right Lateral Ventricle Volume ({unit})",
    "中脑": "Midbrain Volume ({unit})",
    "小脑（左）": "Left Cerebellum Volume ({unit})",
    "小脑（右）": "Right Cerebellum Volume ({unit})",
    "胼胝体": "Corpus Callosum Volume ({unit})",
    "壳核（左）": "Left Putamen Volume ({unit})",
    "壳核（右）": "Right Putamen Volume ({unit})",
    "丘脑（左）": "Left Thalamus Volume ({unit})",
    "丘脑（右）": "Right Thalamus Volume ({unit})",
    "第三脑室": "Third Ventricle Volume ({unit})",
    "第四脑室": "Fourth Ventricle Volume ({unit})",
}

In [14]:
def plot_all_vs_ga(
    csv_path,
    out_dir,
    ga_col="gestational_age",
    subj_col="subj_names",
    dpi=300,
    y_scale=1.0,          # ml->mm^3: 1000.0；不缩放: 1.0
    unit_if_scaled="mm³", # y_scale==1000 时显示 mm³
    unit_if_raw="ml",     # y_scale==1 时显示 ml
):
    df = pd.read_csv(csv_path)
    os.makedirs(out_dir, exist_ok=True)

    # GA 转数值并过滤
    df[ga_col] = pd.to_numeric(df[ga_col], errors="coerce")
    df = df.dropna(subset=[ga_col]).reset_index(drop=True)

    # 需要画的 y 列：除了 subj_names / gestational_age / subj_key 之外
    drop_cols = {subj_col, ga_col, "subj_key"}
    y_cols = [c for c in df.columns if c not in drop_cols]

    # 单位文字（仅用于显示）
    unit = unit_if_scaled if float(y_scale) != 1.0 else unit_if_raw
    x_label = COLNAME_EN_MAP.get(ga_col, ga_col)

    # ---- 1) 不带拟合线 ----
    out_nofit = os.path.join(out_dir, "nofit")
    os.makedirs(out_nofit, exist_ok=True)

    for y_col in y_cols:
        y = pd.to_numeric(df[y_col], errors="coerce") * float(y_scale)
        x = df[ga_col]

        m = np.isfinite(x.to_numpy()) & np.isfinite(y.to_numpy())
        xx = x.to_numpy(dtype=float)[m]
        yy = y.to_numpy(dtype=float)[m]
        if len(xx) < 2:
            continue

        plt.figure(figsize=(7, 6))
        plt.rcParams["font.sans-serif"] = ["DejaVu Sans"]
        plt.rcParams["axes.unicode_minus"] = False

        plt.scatter(xx, yy, color='skyblue', s=60, alpha=0.8)

        y_tpl = COLNAME_EN_MAP.get(y_col, y_col)
        y_label = y_tpl.format(unit=unit) if "{unit}" in str(y_tpl) else y_tpl

        plt.xlabel(x_label)
        plt.ylabel(y_label)
        plt.title(f"{y_label} vs {x_label}")
        plt.grid(alpha=0.3)

        # 文件名：保持原始列名（含中文）
        safe_y = "".join(ch if ch.isalnum() or ch in "._-（）()" else "_" for ch in y_col)
        out_path = os.path.join(out_nofit, f"{safe_y}_nofit.png")
        plt.savefig(out_path, dpi=dpi, bbox_inches="tight")
        plt.close()

    # ---- 2) 带拟合线 ----
    out_fit = os.path.join(out_dir, "fit")
    os.makedirs(out_fit, exist_ok=True)

    for y_col in y_cols:
        y = pd.to_numeric(df[y_col], errors="coerce") * float(y_scale)
        x = df[ga_col]

        m = np.isfinite(x.to_numpy()) & np.isfinite(y.to_numpy())
        xx = x.to_numpy(dtype=float)[m]
        yy = y.to_numpy(dtype=float)[m]
        if len(xx) < 2:
            continue

        plt.figure(figsize=(7, 6))
        plt.rcParams["font.sans-serif"] = ["DejaVu Sans"]
        plt.rcParams["axes.unicode_minus"] = False

        plt.scatter(xx, yy, color='skyblue', s=60, alpha=0.8)

        z = np.polyfit(xx, yy, 1)
        p = np.poly1d(z)
        xs = np.linspace(xx.min(), xx.max(), 200)
        plt.plot(xs, p(xs), "r--", alpha=0.7, label=f"Fit: y={z[0]:.5f}x+{z[1]:.2f}")
        plt.legend()

        y_tpl = COLNAME_EN_MAP.get(y_col, y_col)
        y_label = y_tpl.format(unit=unit) if "{unit}" in str(y_tpl) else y_tpl

        plt.xlabel(x_label)
        plt.ylabel(y_label)
        plt.title(f"{y_label} vs {x_label} (with fit)")
        plt.grid(alpha=0.3)

        safe_y = "".join(ch if ch.isalnum() or ch in "._-（）()" else "_" for ch in y_col)
        out_path = os.path.join(out_fit, f"{safe_y}_fit.png")
        plt.savefig(out_path, dpi=dpi, bbox_inches="tight")
        plt.close()

    print(f"[OK] Saved nofit: {out_nofit}")
    print(f"[OK] Saved fit:   {out_fit}")

In [15]:
csv_dropnan = "/mnt/sda/public/00_Projects/2512_AutoRecon/results/gfy_results/gfy_surface_vol_all_with_age_dropna.csv"
out_dir = "/mnt/sda/public/00_Projects/2512_AutoRecon/results/gfy_results/GA_scatter_all"
# 若你的体积列仍是 ml，但你希望画 mm³：y_scale=1000.0
plot_all_vs_ga(csv_dropnan, out_dir, y_scale=1.0)     # 画 ml
# plot_all_vs_ga(csv_dropnan, out_dir, y_scale=1000.0) # 画 mm³

[OK] Saved nofit: /mnt/sda/public/00_Projects/2512_AutoRecon/results/gfy_results/GA_scatter_all/nofit
[OK] Saved fit:   /mnt/sda/public/00_Projects/2512_AutoRecon/results/gfy_results/GA_scatter_all/fit


In [ ]:
# import os
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt


# def plot_reference_with_new_points(
#     reference_csv="/mnt/sda/public/00_Projects/2512_AutoRecon/results/gfy_results/gfy_surface_vol_all_with_age_dropna.csv",
#     new_df=None,                       # 必须包含 subj_names + gestational_age + 若干变量列
#     out_dir="/mnt/sda/public/00_Projects/2512_AutoRecon/results/gfy_results/GA_overlay_plots",
#     subj_col="subj_names",
#     ga_col="gestational_age",
#     mode="together",                   # "together" or "separate"
#     subject_ids=None,                  # None: 用 new_df 中所有 subj_names；或传 list[str]
#     y_cols=None,                       # None: 自动取 reference 中除ID/GA外的列（与 new_df 取交集）
#     col_en_map=COLNAME_EN_MAP,                   # dict: 列名->英文显示名（仅显示，文件名仍用原列名）
#     dpi=300,
#     y_scale=1.0,              # <<< 新增：新点y与参考y统一缩放（ml->mm³: 1000）
#     unit_if_scaled="mm³",     # <<< 新增
#     unit_if_raw="ml",         # <<< 新增
# ):
#     """
#     对每个变量 y_col：
#       - 画 Reference: y vs GA 的散点 + 线性拟合线（仅 reference）
#       - 叠加 new_df 的点（红色*），并在 legend 标注 Subject-xxx
#     mode:
#       - together: 同一变量一张图，叠加多个 subject
#       - separate: 每个 subject × 每个变量一张图
#     """
#     assert new_df is not None, "new_df 不能为空"

#     ref = pd.read_csv(reference_csv).copy()
#     os.makedirs(out_dir, exist_ok=True)

#     # 数值化 GA
#     ref[ga_col] = pd.to_numeric(ref[ga_col], errors="coerce")
#     new_df = new_df.copy()
#     new_df[ga_col] = pd.to_numeric(new_df[ga_col], errors="coerce")

#     # 选 subject 列表
#     if subject_ids is None:
#         subject_ids = [s for s in new_df[subj_col].dropna().astype(str).unique().tolist()]
#     else:
#         subject_ids = [str(s) for s in subject_ids]

#     # 自动 y_cols：reference 中除 subj/ga/subj_key 外的列，与 new_df 取交集（避免 new_df 没有该列）
#     if y_cols is None:
#         drop_cols = {subj_col, ga_col, "subj_key"}
#         cand_ref = [c for c in ref.columns if c not in drop_cols]
#         cand_new = set([c for c in new_df.columns if c not in drop_cols])
#         y_cols = [c for c in cand_ref if c in cand_new]  # 只画双方都有的列

#     # 标签显示
#     def disp(name):
#         return (col_en_map.get(name, name) if isinstance(col_en_map, dict) else name)

#     xlab = disp(ga_col)

#     # plot helper：画一张图（一个y_col + 一组subject_ids）
#     def _plot_one(y_col, subjects_for_this_plot, save_path):
#         # Reference x/y
#         x_ref = pd.to_numeric(ref[ga_col], errors="coerce").to_numpy(dtype=float)
#         y_ref = pd.to_numeric(ref[y_col], errors="coerce").to_numpy(dtype=float)
#         m = np.isfinite(x_ref) & np.isfinite(y_ref)
#         x_ref, y_ref = x_ref[m], y_ref[m]
#         if len(x_ref) < 2:
#             return

#         plt.figure(figsize=(7.2, 6))
#         plt.rcParams["font.sans-serif"] = ["DejaVu Sans"]  # 若你装了Noto/微软雅黑，可改掉
#         plt.rcParams["axes.unicode_minus"] = False

#         # Reference scatter
#         plt.scatter(x_ref, y_ref, color='skyblue', s=60, alpha=0.8, label="Reference")

#         # Reference fit line (ONLY reference)
#         z = np.polyfit(x_ref, y_ref, 1)
#         p = np.poly1d(z)
#         xs = np.linspace(x_ref.min(), x_ref.max(), 200)
#         plt.plot(xs, p(xs), "--", color="dimgray", alpha=0.75, label=f"Ref Fit: y={z[0]:.5f}x+{z[1]:.2f}")

#         # Overlay new points (NOT in fit)
#         for sid in subjects_for_this_plot:
#             sub = new_df.loc[new_df[subj_col].astype(str) == str(sid)]
#             if len(sub) == 0:
#                 continue
#             # 取第一条（若同一sid多行）
#             row = sub.iloc[0]
#             x_new = pd.to_numeric(pd.Series([row[ga_col]]), errors="coerce").iloc[0]
#             y_new = pd.to_numeric(pd.Series([row[y_col]]), errors="coerce").iloc[0]
#             if not (np.isfinite(x_new) and np.isfinite(y_new)):
#                 continue

#             plt.scatter(
#                 x_new, y_new,
#                 color="red", s=200, marker="*", alpha=1.0,
#                 label=f"Subject-{sid}"
#             )

#         plt.xlabel(xlab)
#         plt.ylabel(disp(y_col))
#         plt.title(f"{disp(y_col)} vs {xlab}")
#         plt.grid(alpha=0.3)
#         # plt.plot(xs, p(xs), "r--", alpha=0.7, label=f"Fit: y={z[0]:.5f}x+{z[1]:.2f}")
        
#         plt.legend()

#         plt.savefig(save_path, dpi=dpi, bbox_inches="tight")
#         plt.close()

#     # 开始批量绘图
#     if mode not in ["together", "separate"]:
#         raise ValueError('mode must be "together" or "separate"')

#     for y_col in y_cols:
#         # 文件名：保持原始列名（可能含中文）
#         safe_y = "".join(ch if ch.isalnum() or ch in "._-（）() " else "_" for ch in y_col).strip()

#         if mode == "together":
#             save_path = os.path.join(out_dir, f"{safe_y}_overlay_multi.png")
#             _plot_one(y_col, subject_ids, save_path)

#         else:  # separate
#             for sid in subject_ids:
#                 subj_dir = os.path.join(out_dir, str(sid))
#                 os.makedirs(subj_dir, exist_ok=True)
#                 save_path = os.path.join(subj_dir, f"{safe_y}_overlay_{sid}.png")
#                 _plot_one(y_col, [sid], save_path)

#     print(f"[OK] Saved plots to: {out_dir}")


In [7]:
njgl_df = pd.read_csv("/mnt/sda/public/00_Projects/2512_AutoRecon/results/njgl_analysis/suface_vol_age_all.csv")

In [ ]:

# subj_dir_list = ["01", "02", "03", "06", "07", "08", "09", "10", "11_zwy", "12", "13"]
# # subj_dir_list = ["01", "02"]
# plot_reference_with_new_points(
#     new_df=njgl_df,
#     mode="separate",
#     subject_ids=subj_dir_list,
#     out_dir="/mnt/sda/public/00_Projects/2512_AutoRecon/results/njgl_analysis/GA_overlay_plots_separate",
#     col_en_map=COLNAME_EN_MAP
# )

[OK] Saved plots to: /mnt/sda/public/00_Projects/2512_AutoRecon/results/njgl_analysis/GA_overlay_plots_separate


In [42]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def plot_reference_with_new_points(
    reference_csv="/mnt/sda/public/00_Projects/2512_AutoRecon/results/gfy_results/gfy_surface_vol_all_with_age_dropna.csv",
    new_df=None,
    out_dir="/mnt/sda/public/00_Projects/2512_AutoRecon/results/gfy_results/GA_overlay_plots",
    subj_col="subj_names",
    ga_col="gestational_age",
    mode="together",          # "together" or "separate"
    subject_ids=None,
    y_cols=None,
    col_en_map=None,          # 例如 COLNAME_EN_MAP
    dpi=300,
    y_scale=1.0,              # <<< 新增：新点y与参考y统一缩放（ml->mm³: 1000）
    unit_if_scaled="mm³",     # <<< 新增
    unit_if_raw="ml",         # <<< 新增
):
    assert new_df is not None, "new_df 不能为空"

    ref = pd.read_csv(reference_csv).copy()
    os.makedirs(out_dir, exist_ok=True)

    # 单位显示
    unit = unit_if_scaled if float(y_scale) != 1.0 else unit_if_raw

    # 数值化 GA
    ref[ga_col] = pd.to_numeric(ref[ga_col], errors="coerce")
    new_df = new_df.copy()
    new_df[ga_col] = pd.to_numeric(new_df[ga_col], errors="coerce")

    # subjects
    if subject_ids is None:
        subject_ids = [s for s in new_df[subj_col].dropna().astype(str).unique().tolist()]
    else:
        subject_ids = [str(s) for s in subject_ids]

    # y_cols：reference 与 new_df 取交集（避免 new_df 没有该列）
    if y_cols is None:
        drop_cols = {subj_col, ga_col, "subj_key"}
        cand_ref = [c for c in ref.columns if c not in drop_cols]
        cand_new = set([c for c in new_df.columns if c not in drop_cols])
        y_cols = [c for c in cand_ref if c in cand_new]

    # 显示名：支持 {unit} 占位符
    def disp(colname: str) -> str:
        if isinstance(col_en_map, dict) and colname in col_en_map:
            s = col_en_map[colname]
        else:
            s = colname
        if isinstance(s, str) and "{unit}" in s:
            s = s.format(unit=unit)
        return s

    xlab = disp(ga_col)

    def _plot_one(y_col, subjects_for_this_plot, save_path, legend_flag=True, txt_label=True):
        # reference x/y
        x_ref = pd.to_numeric(ref[ga_col], errors="coerce").to_numpy(dtype=float)
        y_ref = pd.to_numeric(ref[y_col], errors="coerce").to_numpy(dtype=float) * float(y_scale)
        m = np.isfinite(x_ref) & np.isfinite(y_ref)
        x_ref, y_ref = x_ref[m], y_ref[m]
        if len(x_ref) < 2:
            return

        plt.figure(figsize=(7.2, 6))
        plt.rcParams["font.sans-serif"] = ["DejaVu Sans"]
        plt.rcParams["axes.unicode_minus"] = False

        # Reference scatter
        plt.scatter(x_ref, y_ref, color='skyblue', s=60, alpha=0.8, label="Reference")

        # Reference fit line (ONLY reference)
        z = np.polyfit(x_ref, y_ref, 1)
        p = np.poly1d(z)
        xs = np.linspace(x_ref.min(), x_ref.max(), 200)
        plt.plot(xs, p(xs), "--", color="dimgray", alpha=0.8,
                 label=f"Ref Fit: y={z[0]:.5f}x+{z[1]:.2f}")

        # Overlay new points (NOT in fit)
        for sid in subjects_for_this_plot:
            sub = new_df.loc[new_df[subj_col].astype(str) == str(sid)]
            if len(sub) == 0:
                continue
            row = sub.iloc[0]
            x_new = pd.to_numeric(pd.Series([row[ga_col]]), errors="coerce").iloc[0]
            y_new = pd.to_numeric(pd.Series([row[y_col]]), errors="coerce").iloc[0] * float(y_scale)
            if not (np.isfinite(x_new) and np.isfinite(y_new)):
                continue
            if legend_flag:
                plt.scatter(
                    x_new, y_new,
                    color="red", s=200, marker="*", alpha=1.0,
                    label=f"Subject-{sid}"
                )
            else:
                plt.scatter(
                    x_new, y_new,
                    color="orange", s=90, alpha=0.95, marker="*", edgecolors="k", linewidths=0.6
                )
                # plt.legend()
                if txt_label:
                    plt.annotate(
                        f"{sid}",
                        (x_new, y_new),
                        xytext=(5, 5),
                        textcoords="offset points"
                    )  

        plt.xlabel(xlab)
        plt.ylabel(disp(y_col))
        plt.title(f"{disp(y_col)} vs {xlab}")
        plt.grid(alpha=0.3)
        if legend_flag:
            plt.legend()       

        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=dpi, bbox_inches="tight")
        plt.close()

    if mode not in ["together", "separate"]:
        raise ValueError('mode must be "together" or "separate"')

    for y_col in y_cols:
        safe_y = "".join(ch if ch.isalnum() or ch in "._-（）() " else "_" for ch in y_col).strip()

        if mode == "together":
            save_dir = f"{out_dir}/multi"
            save_path = os.path.join(save_dir, f"{safe_y}_overlay_multi.png")
            _plot_one(y_col, subject_ids, save_path, legend_flag=False)
            save_path = os.path.join(save_dir, f"{safe_y}_overlay_multi_unlabel.png")
            _plot_one(y_col, subject_ids, save_path, legend_flag=False, txt_label=False)

        else:  # separate: 每个个体一个文件夹
            for sid in subject_ids:
                subj_dir = os.path.join(out_dir, str(sid))
                os.makedirs(subj_dir, exist_ok=True)
                save_path = os.path.join(subj_dir, f"{safe_y}_overlay_{sid}.png")
                _plot_one(y_col, [sid], save_path)

    print(f"[OK] Saved plots to: {out_dir}")


In [24]:
njgl_df = pd.read_csv("/mnt/sda/public/00_Projects/2512_AutoRecon/results/njgl_analysis/suface_vol_age_all.csv")
subj_dir_list = ["01", "02", "03", "06", "07", "08", "09", "10", "11_zwy", "12", "13"]
# subj_dir_list = ["01", "02"]
plot_reference_with_new_points(
    new_df=njgl_df,
    mode="separate",
    subject_ids=subj_dir_list,
    out_dir="/mnt/sda/public/00_Projects/2512_AutoRecon/results/njgl_analysis/GA_overlay_plots_separate",
    col_en_map=COLNAME_EN_MAP
)

[OK] Saved plots to: /mnt/sda/public/00_Projects/2512_AutoRecon/results/njgl_analysis/GA_overlay_plots_separate


In [43]:
plot_reference_with_new_points(
    new_df=njgl_df,
    mode="together",
    subject_ids=subj_dir_list,
    out_dir="/mnt/sda/public/00_Projects/2512_AutoRecon/results/njgl_analysis/GA_overlay_plots_separate",
    col_en_map=COLNAME_EN_MAP
)

[OK] Saved plots to: /mnt/sda/public/00_Projects/2512_AutoRecon/results/njgl_analysis/GA_overlay_plots_separate
